# Learned Coupling-Matrix Comparison

These matrices are averages over 12 fits. Rows are recipient
counties, columns are source counties, and every row sums to one.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")

ROOT = find_repo_root()
TABLE_DIR = ROOT / "results" / "tables"
MATRIX_DIR = ROOT / "results" / "matrices"

In [ ]:
neighbor = pd.read_csv(MATRIX_DIR / "neighbor_network_learned_matrix_mean.csv", index_col=0)
inferred = pd.read_csv(MATRIX_DIR / "data_inferred_network_learned_matrix_mean.csv", index_col=0)
difference = pd.read_csv(MATRIX_DIR / "data_inferred_minus_neighbor_learned_matrix.csv", index_col=0)
diagnostics = pd.read_csv(MATRIX_DIR / "all64_learned_matrix_distance_summary.csv")
diagnostics.T

In [ ]:
checks = pd.DataFrame({
    "neighbor_row_sum": neighbor.sum(axis=1),
    "inferred_row_sum": inferred.sum(axis=1),
})
checks.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
panels = [(neighbor, "Neighbor Network", "viridis", 0, 1),
          (inferred, "Data-Inferred Network", "viridis", 0, 1)]
limit = np.abs(difference.to_numpy()).max()
panels.append((difference, "Data-Inferred minus Neighbor", "RdBu_r", -limit, limit))
for axis, (matrix, title, cmap, vmin, vmax) in zip(axes, panels):
    image = axis.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    axis.set_title(title)
    axis.set_xlabel("Source county index")
    axis.set_ylabel("Recipient county index")
    fig.colorbar(image, ax=axis, fraction=0.046)
fig.tight_layout()